In [27]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-chinese")
model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-chinese")

Some weights of the model checkpoint at google-bert/bert-base-chinese were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [28]:
def get_poem_vector(poem):
    inputs = tokenizer(poem, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs).logits
    # Average the token embeddings to get a single vector for the poem
    poem_vector = outputs.mean(dim=1)
    return poem_vector

In [32]:
def precompute_and_save_vocab_embeddings(tokenizer, model, batch_size=64, save_path="vocab_embeddings.npy"):
    vocab = tokenizer.get_vocab()
    words = list(vocab.keys())
    embeddings = []

    # Process words in batches
    for i in range(0, len(words), batch_size):
        batch_words = words[i:i + batch_size]
        inputs = tokenizer(batch_words, return_tensors="pt", padding=True, truncation=True)
        
        with torch.no_grad():
            outputs = model(**inputs).logits

        # Compute mean of token embeddings for each word in the batch
        for j, word in enumerate(batch_words):
            word_vector = outputs[j].mean(dim=0).cpu().numpy()  # Mean pooling, then move back to CPU
            embeddings.append(word_vector)

    # Save words and vectors as separate files or a single dictionary
    np.save(save_path, {"words": words, "embeddings": np.array(embeddings)})
    print(f"Vocabulary embeddings saved to {save_path}")

# Example usage
# Assuming 'tokenizer' and 'model' are already defined
precompute_and_save_vocab_embeddings(tokenizer, model, batch_size=64, save_path="vocab_embeddings.npy")

KeyboardInterrupt: 

In [ ]:
words = list(vocab_embeddings.keys())
vectors = np.array(list(vocab_embeddings.values()))

# Save words and vectors separately
np.save("vocab_words.npy", words)
np.save("vocab_vectors.npy", vectors)

KeyboardInterrupt: 

In [ ]:
def find_closest_word_from_precomputed(target_vector, vocab_embeddings):
    closest_word = None
    min_distance = float("inf")

    for word, embedding in vocab_embeddings.items():
        distance = np.linalg.norm(target_vector - embedding)

        if distance < min_distance:
            min_distance = distance
            closest_word = word

    return closest_word

In [16]:
poem1 = "啊啊啊啊啊啊啊。"
poem2 = "哦哦哦哦哦哦哦。"

poem_vector1 = get_poem_vector(poem1)
poem_vector2 = get_poem_vector(poem2)

combined_vector = poem_vector1 + poem_vector2
combined_vector_np = combined_vector.numpy()